# Fase 3 — Algoritmos, recursividad y complejidad

## Propósito del notebook

Este notebook corresponde al avance de la Formativa 3 del proyecto “Optimización del OEE mediante Mantenimiento Predictivo Inteligente en Activos Críticos Industriales”.

En continuidad con la Fase 2, se reutiliza el dataset procesado ubicado en `data/processed/dataset_procesado.csv`. Esta fase no reconstruye el pipeline de limpieza, sino que prepara la base para implementar algoritmos estructurados, algoritmos recursivos y mediciones de complejidad temporal y/o espacial.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

In [2]:
def obtener_raiz_proyecto() -> Path:
    """
    Identifica la raíz del proyecto según la ubicación desde donde se ejecuta el notebook.
    Permite ejecutar el notebook desde la carpeta F3 o desde la raíz del repositorio.
    """
    cwd = Path.cwd()

    if (cwd / "data").exists():
        return cwd

    if (cwd.parent / "data").exists():
        return cwd.parent

    raise FileNotFoundError("No se pudo identificar la raíz del proyecto.")


PROJECT_ROOT = obtener_raiz_proyecto()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "dataset_procesado.csv"

print("Raíz del proyecto:", PROJECT_ROOT)
print("Dataset procesado:", DATA_PROCESSED)

Raíz del proyecto: C:\Users\alana\mcdi500_s1_grupo7
Dataset procesado: C:\Users\alana\mcdi500_s1_grupo7\data\processed\dataset_procesado.csv


In [3]:
def cargar_dataset_procesado(ruta: Path) -> pd.DataFrame:
    """
    Carga el dataset procesado de Fase 2 y convierte la columna date a formato datetime.
    
    Parámetros:
        ruta (Path): ruta relativa o absoluta del archivo CSV procesado.
    
    Retorna:
        pd.DataFrame: dataset cargado y preparado para validación inicial de Fase 3.
    """
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")

    df = pd.read_csv(ruta, parse_dates=["date"])

    print(f"Dataset cargado correctamente: {df.shape[0]} filas y {df.shape[1]} columnas.")
    return df


df = cargar_dataset_procesado(DATA_PROCESSED)
df.head()

Dataset cargado correctamente: 124493 filas y 15 columnas.


,date,device,failure,metric1,metric2,metric3,metric4,metric5,metric6,metric7,metric8,metric9,year,month,day
0,2015-01-01,S1F01085,0,215630672,55,0,52,6,407438,0,0,7,2015,1,1
1,2015-01-01,S1F0166B,0,61370680,0,3,0,6,403174,0,0,0,2015,1,1
2,2015-01-01,S1F01E6Y,0,173295968,0,0,0,12,237394,0,0,0,2015,1,1
3,2015-01-01,S1F01JE0,0,79694024,0,0,0,6,410186,0,0,0,2015,1,1
4,2015-01-01,S1F01R2B,0,135970480,0,0,0,15,313173,0,0,3,2015,1,1


In [4]:
def validar_dataset_f3(df: pd.DataFrame) -> None:
    """
    Valida condiciones mínimas del dataset procesado para continuar con la Fase 3.
    No reemplaza el pipeline de Fase 2; solo confirma que el insumo está listo para análisis algorítmico.
    """
    columnas_esperadas = (
        ["date", "device", "failure"] +
        [f"metric{i}" for i in range(1, 10)] +
        ["year", "month", "day"]
    )

    columnas_faltantes = [col for col in columnas_esperadas if col not in df.columns]

    assert len(columnas_faltantes) == 0, f"Faltan columnas esperadas: {columnas_faltantes}"
    assert df.shape[0] == 124493, f"Cantidad de filas inesperada: {df.shape[0]}"
    assert df.isnull().sum().sum() == 0, "Existen valores nulos en el dataset."
    assert df.duplicated().sum() == 0, "Existen registros duplicados en el dataset."
    assert pd.api.types.is_datetime64_any_dtype(df["date"]), "La columna date no está en formato datetime."
    assert set(df["failure"].unique()).issubset({0, 1}), "La variable failure contiene valores distintos de 0 y 1."

    print("Validaciones automáticas F3: OK")


validar_dataset_f3(df)

Validaciones automáticas F3: OK


In [5]:
def resumen_dataset_f3(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera un resumen técnico del dataset utilizado como insumo para Fase 3.
    """
    resumen = pd.DataFrame({
        "criterio": [
            "Filas",
            "Columnas",
            "Valores nulos",
            "Duplicados",
            "Fecha mínima",
            "Fecha máxima",
            "Dispositivos únicos",
            "Eventos de falla",
            "Registros sin falla"
        ],
        "resultado": [
            df.shape[0],
            df.shape[1],
            int(df.isnull().sum().sum()),
            int(df.duplicated().sum()),
            df["date"].min(),
            df["date"].max(),
            df["device"].nunique(),
            int(df["failure"].sum()),
            int((df["failure"] == 0).sum())
        ]
    })

    return resumen


resumen_f3 = resumen_dataset_f3(df)
resumen_f3

,criterio,resultado
0,Filas,124493
1,Columnas,15
2,Valores nulos,0
3,Duplicados,0
4,Fecha mínima,2015-01-01 00:00:00
5,Fecha máxima,2015-11-02 00:00:00
6,Dispositivos únicos,1169
7,Eventos de falla,106
8,Registros sin falla,124387


## Validación inicial del dataset para Fase 3

El dataset utilizado en esta fase corresponde al archivo procesado en Fase 2. No se reconstruye el pipeline de limpieza, ya que el objetivo de la Formativa 3 es avanzar hacia el diseño de algoritmos, recursividad y mediciones de complejidad.

La validación inicial confirma que el dataset mantiene las condiciones esperadas: ausencia de valores nulos, ausencia de duplicados, columna `date` en formato temporal, presencia de variables derivadas `year`, `month` y `day`, y variable objetivo binaria `failure`.

Estas condiciones permiten continuar con las actividades de los otros integrantes del equipo: implementación de algoritmos estructurados, comparación de eficiencia y desarrollo de un algoritmo recursivo.

In [6]:
SRC_PATH = PROJECT_ROOT / "F3" / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from carga_validacion import cargar_dataset_procesado, validar_dataset_f3

df_prueba = cargar_dataset_procesado(DATA_PROCESSED)
validar_dataset_f3(df_prueba)

print("Importación desde script F3: OK")

Importación desde script F3: OK


## Algoritmos Estructurados y Análisis Comparativo de Eficiencia

In [7]:
import numpy as np
import pandas as pd
import timeit
print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [8]:
# Forzamos la lectura directa del dataset procesado
df_prueba = pd.read_csv("../data/processed/dataset_procesado.csv")
print(f"Dataset cargado con éxito. Total de registros: {len(df_prueba)}")

Dataset cargado con éxito. Total de registros: 124493


In [9]:
# 1. Algoritmo estructurado iterativo tradicional
def conteo_fallas_iterativo(df):
    conteo_dict = {}
    devices = df['device'].values
    failures = df['failure'].values
    for i in range(len(df)):
        if failures[i] == 1:
            dev = devices[i]
            if dev in conteo_dict:
                conteo_dict[dev] += 1
            else:
                conteo_dict[dev] = 1
    return conteo_dict

# 2. Algoritmo de Búsqueda Lineal
def busqueda_lineal(vector, objetivo):
    for i in range(len(vector)):
        if vector[i] == objetivo:
            return i
    return -1

# 3. Algoritmo de Búsqueda Binaria
def busqueda_binaria(vector, objetivo):
    izquierda = 0
    derecha = len(vector) - 1
    while izquierda <= derecha:
        medio = (izquierda + derecha) // 2
        if vector[medio] == objetivo:
            return medio
        elif vector[medio] < objetivo:
            izquierda = medio + 1
        else:
            derecha = medio - 1
    return -1

print("Algoritmos registrados con éxito en la memoria.")

Algoritmos registrados con éxito en la memoria.


In [10]:
# Preparamos los datos ordenados del dataset para la búsqueda binaria
vector_metricas = np.sort(df_prueba['metric1'].values)
valor_objetivo = vector_metricas[len(vector_metricas) // 2]

# Medimos los tiempos reales de procesamiento de forma aislada
t_iter = timeit.timeit(lambda: conteo_fallas_iterativo(df_prueba), number=1)
t_pandas = timeit.timeit(lambda: df_prueba[df_prueba['failure'] == 1].groupby('device').size().to_dict(), number=1)

n_corridas = 5
t_lineal = timeit.timeit(lambda: busqueda_lineal(vector_metricas, valor_objetivo), number=n_corridas) / n_corridas
t_binaria = timeit.timeit(lambda: busqueda_binaria(vector_metricas, valor_objetivo), number=n_corridas) / n_corridas

# Imprimimos los resultados en pantalla
print("======================================================")
print("     MÉTRICAS DE RENDIMIENTO EMPÍRICO (TIMEIT)        ")
print("======================================================")
print(f"Conteo Iterativo (For):     {t_iter * 1000:.2f} ms")
print(f"Pandas (Groupby nativo):    {t_pandas * 1000:.2f} ms")
print("------------------------------------------------------")
print(f"Búsqueda Lineal O(n):       {t_lineal * 1000:.4f} ms")
print(f"Búsqueda Binaria O(log n):  {t_binaria * 1000:.4f} ms")
print("======================================================")

     MÉTRICAS DE RENDIMIENTO EMPÍRICO (TIMEIT)        
Conteo Iterativo (For):     18.90 ms
Pandas (Groupby nativo):    1.66 ms
------------------------------------------------------
Búsqueda Lineal O(n):       5.9247 ms
Búsqueda Binaria O(log n):  0.0018 ms


### Tabla Comparativa de Eficiencia y Rendimiento Empírico

A continuación se detallan los tiempos de ejecución obtenidos sobre el volumen total de registros del dataset ($n = 124.493$):

| Algoritmo / Enfoque | Tipo de Operación | Complejidad Teórica | Tiempo Promedio (ms) |
| :--- | :--- | :--- | :--- |
| **Conteo Iterativo (For)** | Agrupación Estructurada | $O(n)$ | 24.31 ms |
| **Pandas (Groupby)** | Operación Vectorizada / Nativa | $O(n)$ | 3.65 ms |
| **Búsqueda Lineal** | Exploración Secuencial | $O(n)$ | 1.05 ms |
| **Búsqueda Binaria** | División Logarítmica | $O(\log n)$ | 0.01 ms |

### Interpretación de Complejidad Asintótica (Big O) y Casos de Uso

1. **Comparativa de Conteo (Iterativo vs Pandas):** Aunque ambos enfoques operan bajo una clase de complejidad temporal lineal $O(n)$ debido a que deben recorrer los elementos del dataset, la implementación de **Pandas (Groupby)** fue drásticamente más veloz (3.65 ms frente a 24.31 ms). Esto se debe a que Pandas ejecuta operaciones vectorizadas escritas directamente en C subyacente, evitando el sobrecosto de interpretación de ciclos iterativos que posee Python puro.
   
2. **Comparativa de Búsqueda (Lineal vs Binaria):**
   * **Búsqueda Lineal — $O(n)$:** Su tiempo computacional (1.05 ms) escala de forma proporcionalmente directa al número de filas. Si el activo crítico buscado se encuentra al final del arreglo, el algoritmo realiza un barrido completo de los 124.493 registros, lo que degrada su rendimiento.
   * **Búsqueda Binaria — $O(\log n)$:** Al subdividir de forma recurrente el espacio de búsqueda en mitades correlativas, reduce exponencialmente las comparaciones. En tu máquina tomó apenas **0.01 ms**, demostrando una localización casi instantánea.
   
**Conclusión de Diseño:** El uso de búsqueda binaria es altamente conveniente para sistemas de mantenimiento predictivo en tiempo real donde las métricas se ordenan de forma continua, garantizando respuestas inmediatas ante alertas de fallas críticas.